# Compute-Aware Traffic Forecasting on LargeST-SDCompanion notebook for the ICAISD 2026 submission (Track 1: SustainableTransportation and Smart Cities).**What this measures.** Every traffic-forecasting paper reports MAE. Almostnone report what the MAE cost. This notebook trains a spread of models on thesame data under the same protocol and records, for each one: accuracy, realGPU energy draw during training, inference energy, latency, parameters andFLOPs. The output is a Pareto frontier over accuracy and cost.**Dataset.** [LargeST](https://github.com/liuxu77/LargeST) (Liu et al., NeurIPS2023 Datasets & Benchmarks) -- CalTrans PeMS loop detectors. We use the SanDiego subset: 716 sensors, calendar 2019, resampled to 15-minute bins, 12 stepsin / 12 steps out.**Where to run it.** Works on either Kaggle or Colab; the next cell detectswhich and sets its paths accordingly.*Kaggle (preferred)* -- LargeST is a Kaggle dataset, so it mounts read-only at`/kaggle/input` and there is nothing to download. Add it via **+ Add Input**,search `liuxu77/largest`. Set **Accelerator = GPU T4**, and turn **Internet on**(the repo still has to be cloned). Do not pick P100: its NVML energy counterdoes not exist on Pascal, so energy falls back to power sampling.*Colab* -- needs a Kaggle API token and pulls ~7.8 GB over the network first.**Cost.** The full seven-model sweep is roughly 3-6 hours on a T4; thelightweight models alone are under 30 minutes. Start with `QUICK_SWEEP = True`.

## 1. Environment

In [ ]:
import subprocess, sys

# nvidia-smi is absent on a CPU runtime, and the raw FileNotFoundError that
# raises says nothing about the actual problem. Catch it so the assert below
# gets to deliver the message that tells you what to do.
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,power.max_limit',
                          '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('nvidia-smi not found -- no GPU is attached to this runtime.')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No GPU attached. Runtime > Change runtime type > T4 GPU, then Run all again. '
    'Energy is measured through NVML, so a CPU run cannot produce the paper numbers.')

In [ ]:
# One place where the two hosts differ, so nothing below has to care again.
import os, sys

IN_KAGGLE = os.path.isdir('/kaggle/input')
HOST = 'kaggle' if IN_KAGGLE else 'colab'

# Kaggle names the mount folder after the dataset slug -- usually. Attach it
# through a different UI path and it lands under another name or an extra
# level of nesting ('datasets/...', say). The marker file is what matters,
# so search for that instead of trusting the name.
def _find_ca_dir():
    for base, dirs, files in os.walk('/kaggle/input'):
        if 'ca_meta.csv' in files:
            return base
        dirs[:] = sorted(dirs)[:20]       # datasets are shallow; stay sane
    return None

if IN_KAGGLE:
    ROOT = '/kaggle/working/LargeST'      # writable; /kaggle/input is not
    CA_DIR = _find_ca_dir()               # the dataset, mounted read-only
    OUT = '/kaggle/working'
else:
    ROOT = '/content/LargeST'
    CA_DIR = f'{ROOT}/data/ca'            # downloaded in section 4
    # Colab recycles runtimes and takes /content with them. A 3-6 hour sweep is
    # long enough that this is a matter of when, not if, so results go to Drive.
    # Mounted here rather than at the sweep so it fails now, not four hours in.
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/icaisd'

RESULTS, FIGURES = f'{OUT}/results', f'{OUT}/figures'

# Logs and checkpoints stay on local disk. The engine writes a checkpoint on
# every validation improvement, and routing that through a Drive mount would
# put network I/O in the training loop. Results are small and written once per
# model, so those are the ones worth persisting.
LOGS = f'{ROOT}/logs'

print(f'host={HOST}\nroot={ROOT}\nca_dir={CA_DIR}\nresults={RESULTS}')

if IN_KAGGLE and CA_DIR is None:
    attached = sorted(os.listdir('/kaggle/input'))
    raise SystemExit(
        f'No folder under /kaggle/input contains ca_meta.csv. '
        f'Attached inputs: {attached or "(none)"}\n'
        'Add the dataset with "+ Add Input" > search "liuxu77/largest" > Add, '
        'then re-run this cell.')

In [ ]:
# tables  -> pandas HDF5 support (LargeST ships .h5)
# nvidia-ml-py -> provides the `pynvml` module used for energy measurement
!pip install --quiet tables nvidia-ml-py h5py

import pynvml
pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
try:
    pynvml.nvmlDeviceGetTotalEnergyConsumption(h)
    print('NVML total-energy counter available -- energy measured directly')
except Exception:
    print('NVML total-energy counter missing -- will integrate power samples instead')

## 2. Get LargeST and patch it for a 2026 stackThe repo targets pandas 1.x / PyTorch 1.12. Two calls are hard errors on a 2026image; `greenbench.compat` fixes them and prints what it changed.On Kaggle this needs **Internet on** in the sidebar settings, or the clonefails.

In [ ]:
import os
os.makedirs(os.path.dirname(ROOT), exist_ok=True)
os.chdir(os.path.dirname(ROOT))

if not os.path.isdir(ROOT):
    !git clone --quiet --depth 1 https://github.com/liuxu77/LargeST.git {ROOT}

assert os.path.isdir(ROOT), (
    f'clone failed -- {ROOT} does not exist. On Kaggle, turn Internet on in the '
    'sidebar settings (needs a phone-verified account) and re-run.')

# Everything downstream resolves data paths relative to cwd, so stay here.
os.chdir(ROOT)
for d in ['greenbench', RESULTS, FIGURES, LOGS]:
    os.makedirs(d, exist_ok=True)
print('cwd', os.getcwd())

## 3. Install `greenbench`These cells write out the measurement package. They are generated from thelocal `greenbench/` sources by `build_notebook.py` -- edit the `.py` files andregenerate rather than editing here, or the two will drift.

In [ ]:
%%writefile greenbench/__init__.py
"""greenbench -- compute-aware evaluation of traffic forecasting models.

Companion code for the ICAISD 2026 submission. Import order matters: call
`greenbench.compat.apply_all(repo_root)` and put the LargeST checkout on
sys.path before importing `runner`, which pulls in `src.*`.
"""

__version__ = '0.1.0'

from . import compat, instrument  # safe: no dependency on src.*

__all__ = ['compat', 'instrument']

In [ ]:
%%writefile greenbench/instrument.py
"""Compute/energy instrumentation for the ICAISD 2026 traffic-forecasting study.

Three measurements underpin the paper's efficiency axis:

  * energy  -- real GPU energy draw (joules) over a code region, via NVML
  * flops   -- static forward-pass cost, via torch.utils.flop_counter
  * latency -- inference wall time, via CUDA events

Energy is the primary metric. FLOPs are reported as a hardware-independent
cross-check, but see `count_flops` for why they are not always trustworthy.
"""

import json
import threading
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict

import torch

try:
    import pynvml
    _NVML = True
except ImportError:  # pragma: no cover - depends on runtime
    _NVML = False


# Grid carbon intensity, gram CO2e per kWh.
#
# SOURCED 2026-09-09. LargeST is California data, so we use the EPA eGRID
# subregion CAMX (WECC California) total output CO2e rate: 430.0 lb/MWh from
# eGRID2023 Rev 2 (released 2025-06-12), converted at 0.45359237 kg/lb ->
# 195.04 gCO2e/kWh. The previous value was an uncited placeholder of 400.0,
# which overstated every carbon figure by 2.05x.
DEFAULT_GRID_INTENSITY = 195.04


# --------------------------------------------------------------------------
# energy
# --------------------------------------------------------------------------

class GPUEnergyMeter:
    """Measures GPU energy over a region.

    Prefers NVML's total-energy counter, which is a hardware accumulator and
    needs no sampling. That counter exists on Volta and newer, which covers
    every GPU Colab hands out (T4, L4, A100). Where it is missing we fall back
    to integrating instantaneous power on a background thread, which is
    noisier -- `self.method` records which path was used so the paper can say
    so honestly.
    """

    def __init__(self, device_index=0, sample_interval=0.05):
        self.device_index = device_index
        self.sample_interval = sample_interval
        self.method = "unavailable"
        self.joules = 0.0
        self._handle = None
        self._stop = None
        self._thread = None
        self._samples = []

        if not _NVML:
            return
        try:
            pynvml.nvmlInit()
            self._handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
        except Exception as exc:  # pragma: no cover
            warnings.warn(f"NVML init failed, energy will not be measured: {exc}")
            self._handle = None
            return

        try:
            pynvml.nvmlDeviceGetTotalEnergyConsumption(self._handle)
            self.method = "nvml_total_energy"
        except Exception:
            self.method = "power_integration"

    @property
    def available(self):
        return self._handle is not None

    def _read_total_mj(self):
        return pynvml.nvmlDeviceGetTotalEnergyConsumption(self._handle)

    def _poll(self):
        while not self._stop.is_set():
            try:
                milliwatts = pynvml.nvmlDeviceGetPowerUsage(self._handle)
                self._samples.append((time.time(), milliwatts / 1000.0))
            except Exception:
                pass
            self._stop.wait(self.sample_interval)

    def start(self):
        if not self.available:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        if self.method == "nvml_total_energy":
            self._start_mj = self._read_total_mj()
        else:
            self._samples = []
            self._stop = threading.Event()
            self._thread = threading.Thread(target=self._poll, daemon=True)
            self._thread.start()
        self._t0 = time.time()

    def stop(self):
        if not self.available:
            self.joules = float("nan")
            self.seconds = float("nan")
            return self.joules
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.seconds = time.time() - self._t0

        if self.method == "nvml_total_energy":
            self.joules = (self._read_total_mj() - self._start_mj) / 1000.0
        else:
            self._stop.set()
            self._thread.join(timeout=2.0)
            # trapezoidal integration of power over time
            total = 0.0
            for (t0, p0), (t1, p1) in zip(self._samples, self._samples[1:]):
                total += 0.5 * (p0 + p1) * (t1 - t0)
            self.joules = total
        return self.joules


@contextmanager
def measure_energy(device_index=0):
    """`with measure_energy() as m: ...` then read `m.joules` / `m.seconds`."""
    meter = GPUEnergyMeter(device_index)
    meter.start()
    try:
        yield meter
    finally:
        meter.stop()


def joules_to_kwh(joules):
    return joules / 3.6e6


def carbon_grams(joules, grid_intensity=DEFAULT_GRID_INTENSITY):
    """gCO2e for a given GPU energy draw.

    Counts GPU energy only -- not host CPU, RAM, cooling or PUE. State that
    scoping in the paper; it makes the number a lower bound rather than a
    wrong one.
    """
    return joules_to_kwh(joules) * grid_intensity


# --------------------------------------------------------------------------
# flops
# --------------------------------------------------------------------------

def count_flops(model, sample_input, label=None):
    """Static forward FLOPs for one batch.

    Returns (flops, reliable). `reliable` is False when the counter almost
    certainly under-counted -- most importantly for nn.LSTM, which dispatches
    to a single fused cuDNN/oneDNN kernel that the dispatcher-level counter
    cannot see through. Recurrent baselines therefore get an analytic count
    instead; everything else is measured.
    """
    from torch.utils.flop_counter import FlopCounterMode

    model.eval()
    counter = FlopCounterMode(display=False)
    try:
        with counter:
            with torch.no_grad():
                model(sample_input, label)
        flops = counter.get_total_flops()
    except Exception as exc:
        warnings.warn(f"FLOP counting failed: {exc}")
        return float("nan"), False

    has_rnn = any(isinstance(m, (torch.nn.LSTM, torch.nn.GRU, torch.nn.RNN))
                  for m in model.modules())
    if has_rnn:
        analytic = _analytic_rnn_flops(model, sample_input)
        if analytic > 0:
            return flops + analytic, True
        return flops, False

    # A count of zero means the tracer saw nothing at all.
    return flops, flops > 0


def _analytic_rnn_flops(model, sample_input):
    """Closed-form FLOPs for RNN layers the tracer misses.

    An LSTM layer costs 4 gate matmuls on the input (input_size x hidden) and
    4 on the recurrent state (hidden x hidden), per timestep, per sequence.
    Multiply-accumulate counts as 2 FLOPs.
    """
    total = 0
    batch = sample_input.shape[0]
    nodes = sample_input.shape[2]
    steps = sample_input.shape[1]
    # LargeST models fold the node axis into the batch before the recurrence
    sequences = batch * nodes

    gate_mult = {torch.nn.LSTM: 4, torch.nn.GRU: 3, torch.nn.RNN: 1}
    for module in model.modules():
        for cls, gates in gate_mult.items():
            if isinstance(module, cls):
                h = module.hidden_size
                for layer in range(module.num_layers):
                    in_size = module.input_size if layer == 0 else h
                    per_step = 2 * gates * (in_size * h + h * h)
                    total += per_step * steps * sequences
                break
    return total


# --------------------------------------------------------------------------
# latency
# --------------------------------------------------------------------------

def measure_latency(model, sample_input, label=None, warmup=10, iters=50):
    """Median forward-pass latency in ms, timed with CUDA events.

    Median rather than mean because Colab hosts are shared and the tail is
    contaminated by other tenants.
    """
    model.eval()
    device = next(model.parameters()).device
    if device.type != "cuda":
        times = []
        with torch.no_grad():
            for _ in range(warmup):
                model(sample_input, label)
            for _ in range(iters):
                t0 = time.perf_counter()
                model(sample_input, label)
                times.append((time.perf_counter() - t0) * 1000)
        times.sort()
        return times[len(times) // 2]

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)
    times = []
    with torch.no_grad():
        for _ in range(warmup):
            model(sample_input, label)
        torch.cuda.synchronize()
        for _ in range(iters):
            starter.record()
            model(sample_input, label)
            ender.record()
            torch.cuda.synchronize()
            times.append(starter.elapsed_time(ender))
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------
# result record
# --------------------------------------------------------------------------

@dataclass
class RunRecord:
    """One (model, dataset, seed) run. Serialised to results/*.json."""
    model: str = ""
    dataset: str = ""
    seed: int = 0
    params: int = 0

    # accuracy, averaged over the 12-step horizon
    mae: float = float("nan")
    rmse: float = float("nan")
    mape: float = float("nan")
    horizon_mae: list = field(default_factory=list)
    horizon_rmse: list = field(default_factory=list)
    horizon_mape: list = field(default_factory=list)

    # cost
    train_joules: float = float("nan")
    train_seconds: float = float("nan")
    epochs_run: int = 0
    epoch_seconds: float = float("nan")
    infer_joules_per_1k: float = float("nan")
    latency_ms: float = float("nan")
    flops_per_sample: float = float("nan")
    flops_reliable: bool = False

    energy_method: str = ""
    gpu_name: str = ""
    notes: str = ""

    @property
    def train_kwh(self):
        return joules_to_kwh(self.train_joules)

    def carbon_g(self, grid_intensity=DEFAULT_GRID_INTENSITY):
        return carbon_grams(self.train_joules, grid_intensity)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as fh:
            json.dump(asdict(self), fh, indent=2)

    @staticmethod
    def load(path):
        with open(path, encoding="utf-8") as fh:
            return RunRecord(**json.load(fh))


def gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "cpu"

In [ ]:
%%writefile greenbench/compat.py
"""Patches LargeST for a modern Python stack.

The repo targets PyTorch 1.12 / pandas 1.x (2023). Colab in 2026 ships pandas
2.x and numpy 2.x, where a couple of the repo's calls are hard errors rather
than warnings. Call `apply_all(repo_root)` once after cloning, before importing
anything from `src`.

Each patch is idempotent and logs what it touched, so a run that silently
skipped a fix is visible rather than mysterious.
"""

import os
import re


def _sub_in_file(path, pattern, replacement, label):
    if not os.path.exists(path):
        return f'SKIP {label}: {path} missing'
    with open(path, encoding='utf-8') as fh:
        src = fh.read()
    new, n = re.subn(pattern, replacement, src)
    if n == 0:
        return f'OK   {label}: nothing to change (already patched?)'
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(new)
    return f'FIX  {label}: {n} replacement(s) in {os.path.basename(path)}'


def patch_pandas_append(repo_root):
    """DataFrame.append was removed in pandas 2.0; use pd.concat.

    Hits generate_data_for_training.py, which builds the multi-year frame with
    `df = df.append(df_tmp)`. Without this the data-generation step dies with
    AttributeError before producing his.npz.
    """
    path = os.path.join(repo_root, 'data', 'generate_data_for_training.py')
    return _sub_in_file(
        path,
        r'df = df\.append\(df_tmp\)',
        'df = pd.concat([df, df_tmp], axis=0) if len(df) else df_tmp',
        'pandas append -> concat')


def patch_np_asarray_adj(repo_root):
    """normalize_adj_mx returns np.matrix via .todense().

    torch.tensor() on an np.matrix keeps the 2-D matrix semantics and newer
    numpy is stricter about it. Returning ndarray avoids a class of shape
    surprises downstream.
    """
    path = os.path.join(repo_root, 'src', 'utils', 'graph_algo.py')
    return _sub_in_file(
        path,
        r'adj = \[a\.astype\(np\.float32\)\.todense\(\) for a in adj\]',
        'adj = [np.asarray(a.astype(np.float32).todense()) for a in adj]',
        'adj todense -> ndarray')


def apply_all(repo_root, verbose=True):
    results = [
        patch_pandas_append(repo_root),
        patch_np_asarray_adj(repo_root),
    ]
    if verbose:
        for line in results:
            print(line)
    return results

In [ ]:
%%writefile greenbench/prepare.py
"""Builds the LargeST-SD subset without blowing up Colab's RAM.

The upstream recipe is: download the whole Kaggle archive, load the full
California frame, slice out District 11. That frame is 8,600 sensors x 105,120
five-minute steps; in float64 it is roughly 7 GB, which a free Colab instance
does not have. Since we only ever want 716 of those columns, this module reads
the file in row blocks and keeps just the columns it needs -- peak memory is a
few hundred MB.

The output is byte-identical in intent to running data/ca/process_ca_his.ipynb
followed by data/sd/generate_sd_dataset.ipynb: District 11 sensors, resampled
to 15 minutes, NaNs zero-filled.
"""

import os

import numpy as np
import pandas as pd


def _decode_index(axis, raw):
    """Rebuild a DatetimeIndex from a raw HDF5 axis, honouring its stored unit.

    The integers under axis1 are counts since the epoch, but the unit is a
    property of the pandas that wrote the file: 1.x always wrote
    datetime64[ns], 3.x writes datetime64[us]. LargeST was published from
    pandas 1.x, so a nanosecond assumption reads the archive correctly and
    then quietly fails on anything regenerated locally -- a 2019 index comes
    back as January 1970, and because it is still monotonic nothing raises.
    The resample simply collapses 2,880 rows into 2. So take the unit the
    file declares rather than assuming one.
    """
    kind = axis.attrs.get('kind', b'datetime64[ns]')
    if isinstance(kind, bytes):
        kind = kind.decode()
    kind = str(kind)
    if kind.startswith('datetime64'):
        # pandas 1.x -- which wrote the published LargeST archive -- stores a
        # bare 'datetime64' with no unit, and numpy will not build a dtype from
        # that. It always meant nanoseconds, so supply the unit it omitted.
        if '[' not in kind:
            kind = 'datetime64[ns]'
        return pd.DatetimeIndex(raw.view(np.dtype(kind)))
    return pd.to_datetime(raw)


def read_hdf_columns(path, wanted_columns, chunk_rows=5000, verbose=True):
    """Read a subset of columns from a pandas HDF5 file, block by block.

    Tries the cheap path first (`pd.read_hdf(columns=...)`, which works for
    table-format files). Falls back to reading the raw h5py datasets, which is
    what fixed-format files require -- they support no partial reads at all
    through pandas.
    """
    try:
        df = pd.read_hdf(path, columns=list(wanted_columns))
        if verbose:
            print(f'read {path} via pandas column selection: {df.shape}')
        return df
    except (TypeError, ValueError, NotImplementedError) as exc:
        if verbose:
            print(f'pandas column selection unavailable ({exc.__class__.__name__}), '
                  f'falling back to chunked h5py read')

    import h5py

    with h5py.File(path, 'r') as fh:
        key = next(iter(fh.keys()))
        grp = fh[key]

        def _decode(arr):
            return [v.decode() if isinstance(v, bytes) else str(v) for v in arr]

        # Fixed-format frames store values per dtype block; the column order
        # inside a block follows block*_items, not axis0.
        blocks = sorted(k for k in grp.keys() if k.endswith('_values'))
        if not blocks:
            raise RuntimeError(f'{path}: no *_values datasets under /{key}')

        axis1 = grp['axis1']
        index_raw = axis1[:]
        n_rows = len(index_raw)

        wanted = [str(c) for c in wanted_columns]
        wanted_set = set(wanted)
        collected = {}

        for values_key in blocks:
            items_key = values_key.replace('_values', '_items')
            if items_key not in grp:
                continue
            names = _decode(grp[items_key][:])
            take = [(i, n) for i, n in enumerate(names) if n in wanted_set]
            if not take:
                continue

            positions = [i for i, _ in take]
            labels = [n for _, n in take]
            dset = grp[values_key]

            if dset.shape[0] != n_rows:
                raise RuntimeError(
                    f'{path}: {values_key} is {dset.shape}, expected {n_rows} rows first; '
                    'this pandas version stores blocks transposed')

            out = np.empty((n_rows, len(positions)), dtype=np.float32)
            for start in range(0, n_rows, chunk_rows):
                stop = min(start + chunk_rows, n_rows)
                out[start:stop] = dset[start:stop, :][:, positions]
            for j, label in enumerate(labels):
                collected[label] = out[:, j]
            if verbose:
                print(f'  {values_key}: pulled {len(positions)} columns')

        missing = wanted_set - set(collected)
        if missing:
            raise KeyError(f'{len(missing)} requested sensors not in {path}, '
                           f'e.g. {sorted(missing)[:5]}')

        df = pd.DataFrame({c: collected[c] for c in wanted},
                          index=_decode_index(axis1, index_raw))

    if verbose:
        print(f'read {path} via chunked h5py: {df.shape}')
    return df


def build_sd_subset(ca_dir, sd_dir, year='2019', resample='15min', verbose=True):
    """Produce sd_meta.csv, sd_rn_adj.npy and sd_his_<year>.h5.

    Mirrors the upstream notebooks. Note the resample default: the reference
    experiments use 15-minute bins (STGODE's --tpd default of 96 confirms it),
    so a day is 96 steps. Pass resample=None to keep the raw 5-minute feed,
    but then every steps_per_day setting downstream must change to 288.
    """
    os.makedirs(sd_dir, exist_ok=True)

    ca_meta = pd.read_csv(os.path.join(ca_dir, 'ca_meta.csv'))
    sd_meta = ca_meta[ca_meta.District == 11].reset_index(drop=True)
    sd_meta.to_csv(os.path.join(sd_dir, 'sd_meta.csv'), index=False)
    if verbose:
        print(f'District 11 sensors: {len(sd_meta)}')

    # adjacency: index into the CA matrix by ID2 (the row position in ca_meta)
    id2 = sd_meta.ID2.values.tolist()
    ca_adj = np.load(os.path.join(ca_dir, 'ca_rn_adj.npy'))
    sd_adj = ca_adj[id2][:, id2]
    np.save(os.path.join(sd_dir, 'sd_rn_adj.npy'), sd_adj)
    if verbose:
        print(f'adjacency {ca_adj.shape} -> {sd_adj.shape}')

    sensor_ids = sd_meta.ID.astype(str).values.tolist()
    raw = os.path.join(ca_dir, f'ca_his_raw_{year}.h5')
    processed = os.path.join(ca_dir, f'ca_his_{year}.h5')
    source = raw if os.path.exists(raw) else processed
    if not os.path.exists(source):
        raise FileNotFoundError(
            f'need {raw} or {processed}; download ca_his_raw_{year}.h5 from Kaggle first')

    sd_his = read_hdf_columns(source, sensor_ids, verbose=verbose)

    if resample and source == raw:
        sd_his = sd_his.resample(resample).mean().round(0)
        if verbose:
            print(f'resampled to {resample}: {sd_his.shape}')

    sd_his = sd_his.fillna(0)
    assert sd_his.isnull().any().sum() == 0, 'nulls survived fillna'

    out = os.path.join(sd_dir, f'sd_his_{year}.h5')
    sd_his.to_hdf(out, key='t', mode='w')
    if verbose:
        print(f'wrote {out}: {sd_his.shape}')
    return sd_his

In [ ]:
%%writefile greenbench/models_lite.py
"""Lightweight baselines -- the cheap end of the accuracy/compute frontier.

LargeST ships twelve baselines, but they are all mid-to-heavy: the cheapest
learned model in the repo is a 2-layer LSTM. The paper's claim is about the
shape of the frontier, so it needs models an order of magnitude below that.
Both models here follow the repo's BaseModel contract exactly -- forward takes
(b, t, n, f) and returns (b, horizon, n, 1) -- so they drop straight into
BaseEngine with no special-casing.

STID follows Shao et al., "Spatial-Temporal Identity: A Simple yet Effective
Baseline for Multivariate Time Series Forecasting" (CIKM 2022). Cite it; do
not present it as ours. Our contribution is the cost-accuracy analysis, not
the architecture.
"""

import torch
import torch.nn as nn

from src.base.model import BaseModel


# The CA history is resampled to 15 minutes in process_ca_his.ipynb, so a day
# is 96 steps, not the 288 you would get from the raw 5-minute feed. If you
# turn the resampling off, pass steps_per_day=288.
STEPS_PER_DAY_15MIN = 96


class HistoricalLast(BaseModel):
    """Repeat the most recent observation across the horizon. Zero parameters.

    The repo ships an equivalent (src/models/hl.py) but it returns all three
    input channels, so it only works when launched with --input_dim 1. Ours
    slices the value channel explicitly, which keeps input_dim=3 uniform
    across the whole lineup and removes a per-model special case from the
    comparison.
    """

    def __init__(self, **args):
        super(HistoricalLast, self).__init__(**args)
        # BaseEngine wants something to hand the optimiser; never used.
        self._unused = nn.Parameter(torch.zeros(1), requires_grad=False)

    def forward(self, input, label=None):  # (b, t, n, f)
        return input[:, [-1], :, 0:1].expand(-1, self.horizon, -1, -1)


class _MLPResidual(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.fc1 = nn.Conv2d(dim, dim, kernel_size=(1, 1), bias=True)
        self.fc2 = nn.Conv2d(dim, dim, kernel_size=(1, 1), bias=True)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.fc2(self.drop(self.act(self.fc1(x))))
        return h + x


class STID(BaseModel):
    """Node/time identity embeddings + an MLP. No graph, no recurrence.

    The whole model is pointwise convolutions over a (b, d, n, 1) tensor, so
    cost scales linearly in node count -- which is the property that matters
    when a city adds sensors.
    """

    def __init__(self, embed_dim=32, node_dim=32, temp_dim=32, layers=3,
                 dropout=0.15, steps_per_day=STEPS_PER_DAY_15MIN,
                 use_time_feats=True, **args):
        super(STID, self).__init__(**args)
        self.use_time_feats = use_time_feats and self.input_dim >= 3
        self.steps_per_day = steps_per_day

        self.node_emb = nn.Parameter(torch.empty(self.node_num, node_dim))
        nn.init.xavier_uniform_(self.node_emb)

        if self.use_time_feats:
            self.tod_emb = nn.Parameter(torch.empty(steps_per_day, temp_dim))
            self.dow_emb = nn.Parameter(torch.empty(7, temp_dim))
            nn.init.xavier_uniform_(self.tod_emb)
            nn.init.xavier_uniform_(self.dow_emb)

        self.series_emb = nn.Conv2d(self.seq_len, embed_dim, kernel_size=(1, 1))

        hidden = embed_dim + node_dim + (2 * temp_dim if self.use_time_feats else 0)
        self.encoder = nn.Sequential(*[_MLPResidual(hidden, dropout) for _ in range(layers)])
        self.head = nn.Conv2d(hidden, self.horizon, kernel_size=(1, 1))

    def forward(self, input, label=None):  # (b, t, n, f)
        b, t, n, _ = input.shape

        # value channel -> (b, t, n, 1) -> conv over the time axis
        x = input[..., 0:1]
        h = self.series_emb(x)  # (b, embed_dim, n, 1)

        feats = [h, self.node_emb.T.unsqueeze(0).unsqueeze(-1).expand(b, -1, -1, 1)]

        if self.use_time_feats:
            # generate_data_for_training stores tod as a fraction of the day
            # and dow as dayofweek/7, both taken at each timestep; index off
            # the most recent one.
            tod = input[:, -1, :, 1]
            dow = input[:, -1, :, 2]
            tod_idx = torch.clamp((tod * self.steps_per_day).round().long(), 0, self.steps_per_day - 1)
            dow_idx = torch.clamp((dow * 7).round().long(), 0, 6)
            feats.append(self.tod_emb[tod_idx].permute(0, 2, 1).unsqueeze(-1))
            feats.append(self.dow_emb[dow_idx].permute(0, 2, 1).unsqueeze(-1))

        h = torch.cat(feats, dim=1)
        h = self.encoder(h)
        out = self.head(h)  # (b, horizon, n, 1)
        return out


class NLinear(BaseModel):
    """A single linear map from the input window to the horizon.

    Deliberately the floor of the frontier: if a model cannot beat this by a
    margin worth its extra joules, that is the finding. The last-value
    subtraction is the normalisation trick from Zeng et al., "Are Transformers
    Effective for Time Series Forecasting?" (AAAI 2023) -- cite it too.

    `individual=True` gives every sensor its own weights (node_num * seq_len *
    horizon params, still tiny); False shares one map across sensors.
    """

    def __init__(self, individual=False, subtract_last=True, **args):
        super(NLinear, self).__init__(**args)
        self.individual = individual
        self.subtract_last = subtract_last

        if individual:
            self.weight = nn.Parameter(torch.empty(self.node_num, self.seq_len, self.horizon))
            self.bias = nn.Parameter(torch.zeros(self.node_num, self.horizon))
            nn.init.xavier_uniform_(self.weight)
        else:
            self.proj = nn.Linear(self.seq_len, self.horizon)

    def forward(self, input, label=None):  # (b, t, n, f)
        x = input[..., 0]  # (b, t, n)

        last = x[:, -1:, :] if self.subtract_last else 0.0
        x = x - last

        x = x.permute(0, 2, 1)  # (b, n, t)
        if self.individual:
            # (b, n, t) x (n, t, h) -> (b, n, h)
            out = torch.einsum("bnt,nth->bnh", x, self.weight) + self.bias
        else:
            out = self.proj(x)

        out = out.permute(0, 2, 1)  # (b, h, n)
        if self.subtract_last:
            out = out + last
        return out.unsqueeze(-1)  # (b, h, n, 1)

In [ ]:
%%writefile greenbench/engine.py
"""BaseEngine subclass that records cost alongside accuracy.

Two things the stock LargeST engine will not give us:

  1. Training energy. We meter each epoch so early stopping is priced in --
     a model that needs 80 epochs to converge is genuinely more expensive
     than one that needs 20, and total-run energy is the only metric that
     captures that.
  2. Test metrics as data. BaseEngine.evaluate('test') writes to a log and
     returns None, so we re-implement it to return the per-horizon arrays.

Everything else -- masking, inverse transform, checkpointing -- is inherited
unchanged, so our numbers stay comparable to published LargeST results.
"""

import time

import numpy as np
import torch

from src.base.engine import BaseEngine
from src.utils.metrics import masked_mape, masked_rmse, compute_all_metrics

from .instrument import GPUEnergyMeter


class InstrumentedEngine(BaseEngine):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_joules = 0.0
        self.train_seconds = 0.0
        self.epochs_run = 0
        self.epoch_joules = []
        self.epoch_seconds = []
        self.test_results = None
        self._meter = GPUEnergyMeter()

    def train(self):
        self._logger.info('Start training (instrumented)!')

        # Historical Last has no trainable parameters, so there is nothing to
        # back-propagate and no training energy to bill it. Rather than fake a
        # gradient to keep the loop uniform, skip it and record a true zero --
        # "this baseline costs nothing to fit" is a fact the paper wants to
        # state, not an artefact to paper over. Still checkpoint, because
        # evaluate_test reloads from disk like every other model.
        if not any(p.requires_grad for p in self.model.parameters()):
            self._logger.info('No trainable parameters -- skipping training, '
                              'train energy recorded as 0 J.')
            self.save_model(self._save_path)
            self.train_joules = 0.0
            self.train_seconds = 0.0
            self.epochs_run = 0
            self.test_results = self.evaluate_test()
            return self.test_results

        wait = 0
        min_loss = np.inf
        run_t0 = time.time()

        for epoch in range(self._max_epochs):
            self._meter.start()
            t1 = time.time()
            mtrain_loss, mtrain_mape, mtrain_rmse = self.train_batch()
            t2 = time.time()
            epoch_j = self._meter.stop()

            self.epoch_joules.append(epoch_j)
            self.epoch_seconds.append(t2 - t1)
            self.epochs_run = epoch + 1

            v1 = time.time()
            mvalid_loss, mvalid_mape, mvalid_rmse = self.evaluate('val')
            v2 = time.time()

            if self._lr_scheduler is None:
                cur_lr = self._lrate
            else:
                cur_lr = self._lr_scheduler.get_last_lr()[0]
                self._lr_scheduler.step()

            message = ('Epoch: {:03d}, Train Loss: {:.4f}, Train RMSE: {:.4f}, '
                       'Train MAPE: {:.4f}, Valid Loss: {:.4f}, Valid RMSE: {:.4f}, '
                       'Valid MAPE: {:.4f}, Train Time: {:.4f}s/epoch, Valid Time: {:.4f}s, '
                       'Train Energy: {:.1f}J, LR: {:.4e}')
            self._logger.info(message.format(
                epoch + 1, mtrain_loss, mtrain_rmse, mtrain_mape,
                mvalid_loss, mvalid_rmse, mvalid_mape,
                (t2 - t1), (v2 - v1), epoch_j, cur_lr))

            if mvalid_loss < min_loss:
                self.save_model(self._save_path)
                self._logger.info('Val loss decrease from {:.4f} to {:.4f}'.format(min_loss, mvalid_loss))
                min_loss = mvalid_loss
                wait = 0
            else:
                wait += 1
                if wait == self._patience:
                    self._logger.info('Early stop at epoch {}, loss = {:.6f}'.format(epoch + 1, min_loss))
                    break

        self.train_seconds = time.time() - run_t0
        # Sum of per-epoch readings rather than one measurement across the
        # whole run: validation passes sit between epochs and should not be
        # billed to training. nansum of an all-NaN list is 0.0, which would
        # report an unmetered run as a free one -- keep it NaN so a missing
        # measurement stays visibly missing.
        self.train_joules = (float(np.nansum(self.epoch_joules))
                             if np.any(np.isfinite(self.epoch_joules))
                             else float('nan'))
        self._logger.info('Total training energy: {:.1f}J over {} epochs ({:.4f} kWh)'.format(
            self.train_joules, self.epochs_run, self.train_joules / 3.6e6))

        self.test_results = self.evaluate_test()
        return self.test_results

    def evaluate_test(self):
        """Like BaseEngine.evaluate('test') but returns the numbers."""
        self.load_model(self._save_path)
        self.model.eval()

        preds, labels = [], []
        with torch.no_grad():
            for X, label in self._dataloader['test_loader'].get_iterator():
                X, label = self._to_device(self._to_tensor([X, label]))
                pred = self.model(X, label)
                pred, label = self._inverse_transform([pred, label])
                preds.append(pred.squeeze(-1).cpu())
                labels.append(label.squeeze(-1).cpu())

        preds = torch.cat(preds, dim=0)
        labels = torch.cat(labels, dim=0)

        mask_value = torch.tensor(0)
        if labels.min() < 1:
            mask_value = labels.min()

        h_mae, h_mape, h_rmse = [], [], []
        for i in range(self.model.horizon):
            mae, mape, rmse = compute_all_metrics(preds[:, i, :], labels[:, i, :], mask_value)
            self._logger.info('Horizon {:d}, Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}'.format(
                i + 1, mae, rmse, mape))
            h_mae.append(mae)
            h_mape.append(mape)
            h_rmse.append(rmse)

        self._logger.info('Average Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}'.format(
            np.mean(h_mae), np.mean(h_rmse), np.mean(h_mape)))

        return {
            'mae': float(np.mean(h_mae)),
            'rmse': float(np.mean(h_rmse)),
            'mape': float(np.mean(h_mape)),
            'horizon_mae': [float(v) for v in h_mae],
            'horizon_rmse': [float(v) for v in h_rmse],
            'horizon_mape': [float(v) for v in h_mape],
        }

    def measure_inference_energy(self, n_batches=30, min_seconds=5.0):
        """Energy per 1000 test samples at inference.

        Deployment cost, as distinct from training cost -- a traffic centre
        trains once and then infers every five minutes forever, so this is
        the number that actually compounds.

        A fixed batch count is not enough at the cheap end of the lineup. Thirty
        batches of NLinear is roughly 10 ms of GPU work, well under what the
        NVML energy counter resolves, so it returns a flat 0 J -- and "this
        model costs no energy to run" is precisely the overclaim this paper
        must not make. So replay the batches until a wall-clock floor has
        passed and divide by the samples actually processed.
        """
        self.model.eval()
        loader = self._dataloader['test_loader']
        meter = GPUEnergyMeter()

        batches = []
        for i, (X, label) in enumerate(loader.get_iterator()):
            if i >= n_batches:
                break
            batches.append(self._to_device(self._to_tensor([X, label])))

        if not batches:
            return float('nan')

        with torch.no_grad():  # warmup
            for X, label in batches[:3]:
                self.model(X, label)

        meter.start()
        samples, passes = 0, 0
        t0 = time.time()
        with torch.no_grad():
            while True:
                for X, label in batches:
                    self.model(X, label)
                    samples += X.shape[0]
                # Without this the CPU races ahead queueing kernels and the
                # elapsed-time test exits before the GPU has done the work.
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                passes += 1
                if time.time() - t0 >= min_seconds:
                    break
        joules = meter.stop()

        self._logger.info(
            'Inference energy: {:.1f}J over {} samples ({} passes, {:.1f}s)'.format(
                joules, samples, passes, time.time() - t0))

        return joules / samples * 1000 if samples else float('nan')

In [ ]:
%%writefile greenbench/runner.py
"""Unified experiment driver.

LargeST's design is one main.py per model, each parsing its own argv. That is
fine from a shell but useless from a notebook, and it makes it easy for the
comparison to drift (different batch sizes, different input_dim). This module
rebuilds every model through one code path with one set of shared settings, so
the only thing varying across runs is the architecture.

Per-model hyperparameters are copied verbatim from the corresponding
experiments/*/main.py defaults -- we are not tuning anyone's baseline down.
"""

import gc
import os
import random

import numpy as np
import torch

from src.base.model import BaseModel
from src.utils.dataloader import load_dataset, load_adj_from_numpy, get_dataset_info
from src.utils.graph_algo import normalize_adj_mx
from src.utils.logging import get_logger
from src.utils.metrics import masked_mae

from .engine import InstrumentedEngine
from .instrument import RunRecord, count_flops, measure_latency, gpu_name
from .models_lite import STID, NLinear, HistoricalLast


class Args:
    """Stand-in for the argparse namespace load_dataset expects."""

    def __init__(self, **kw):
        self.years = '2019'
        self.seq_len = 12
        self.horizon = 12
        self.input_dim = 3
        self.output_dim = 1
        self.bs = 64
        self.__dict__.update(kw)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = False


# --------------------------------------------------------------------------
# model builders -- each returns (model, optimizer_factory, scheduler_factory,
# clip_grad_value). Factories rather than instances so the optimiser is built
# after the model is on-device.
# --------------------------------------------------------------------------

def _build_hl(node_num, adj_path, args, device):
    model = HistoricalLast(node_num=node_num, input_dim=args.input_dim,
                           output_dim=args.output_dim)
    return model, None, None, 0


def _build_nlinear(node_num, adj_path, args, device):
    model = NLinear(node_num=node_num, input_dim=args.input_dim,
                    output_dim=args.output_dim, individual=True)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    return model, opt, None, 5


def _build_stid(node_num, adj_path, args, device):
    model = STID(node_num=node_num, input_dim=args.input_dim,
                 output_dim=args.output_dim, embed_dim=32, node_dim=32,
                 temp_dim=32, layers=3, dropout=0.15)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=1e-4)
    return model, opt, None, 5


def _build_lstm(node_num, adj_path, args, device):
    from src.models.lstm import LSTM
    model = LSTM(node_num=node_num, input_dim=args.input_dim,
                 output_dim=args.output_dim, init_dim=32, hid_dim=64,
                 end_dim=512, layer=2, dropout=0.1)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    return model, opt, None, 5


def _build_stgcn(node_num, adj_path, args, device):
    from src.models.stgcn import STGCN
    Kt, Ks, block_num = 3, 3, 2

    adj_mx = load_adj_from_numpy(adj_path)
    adj_mx = adj_mx - np.eye(node_num)
    gso = normalize_adj_mx(adj_mx, 'scalap')[0]
    gso = torch.tensor(np.asarray(gso), dtype=torch.float32).to(device)

    Ko = args.seq_len - (Kt - 1) * 2 * block_num
    blocks = [[args.input_dim]]
    for _ in range(block_num):
        blocks.append([64, 16, 64])
    blocks.append([128] if Ko == 0 else [128, 128])
    blocks.append([args.horizon])

    model = STGCN(node_num=node_num, input_dim=args.input_dim,
                  output_dim=args.output_dim, gso=gso, blocks=blocks,
                  Kt=Kt, Ks=Ks, dropout=0.5)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=5e-4)
    sched = lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=10, gamma=0.95)
    return model, opt, sched, 0


def _build_gwnet(node_num, adj_path, args, device):
    from src.models.gwnet import GWNET
    adj_mx = load_adj_from_numpy(adj_path)
    adj_mx = normalize_adj_mx(adj_mx, 'doubletransition')
    supports = [torch.tensor(np.asarray(a), dtype=torch.float32).to(device) for a in adj_mx]

    model = GWNET(node_num=node_num, input_dim=args.input_dim,
                  output_dim=args.output_dim, supports=supports, adp_adj=1,
                  dropout=0.3, residual_channels=32, dilation_channels=32,
                  skip_channels=256, end_channels=512)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    return model, opt, None, 5


def _build_sttn(node_num, adj_path, args, device):
    from src.models.sttn import STTN
    adj_mx = load_adj_from_numpy(adj_path)
    adj_mx = normalize_adj_mx(adj_mx, 'doubletransition')
    supports = [torch.tensor(np.asarray(a), dtype=torch.float32).to(device) for a in adj_mx]

    model = STTN(node_num=node_num, input_dim=args.input_dim,
                 output_dim=args.output_dim, device=device, supports=supports,
                 blocks=2, mlp_expand=2, hidden_channels=32, end_channels=512,
                 dropout=0.1)
    opt = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    return model, opt, None, 5


# STGODE is deliberately absent: it precomputes a DTW similarity matrix over
# every sensor pair (fastdtw, O(n^2) pairs), which is hours of CPU for SD=716
# before a single epoch runs. Note the omission in the paper rather than
# pretending the lineup is exhaustive. DCRNN/DGCRN/D2STGNN/AGCRN/ASTGCN/DSTAGNN
# are excluded for a different reason -- they need their own engine subclasses,
# so adding them means duplicating the instrumentation.
MODEL_REGISTRY = {
    'hl': _build_hl,
    'nlinear': _build_nlinear,
    'stid': _build_stid,
    'lstm': _build_lstm,
    'stgcn': _build_stgcn,
    'gwnet': _build_gwnet,
    'sttn': _build_sttn,
}


def run_experiment(model_name, dataset='SD', years='2019', seed=2023, bs=64,
                   max_epochs=100, patience=30, device=None, results_dir='results',
                   log_root='./logs'):
    """Train one model, measure everything, return a RunRecord.

    Patience matches LargeST's 30. An earlier version used 15 to fit a Colab
    session, but the observed runs showed 15 truncating STID -- the model this
    paper argues *for* -- as well as the expensive ones, so it was not the
    neutral shortcut it looked like. Every model in the table must share one
    value; do not lower it for a single run.
    """
    if model_name not in MODEL_REGISTRY:
        raise KeyError(f"unknown model '{model_name}'; have {sorted(MODEL_REGISTRY)}")

    device = torch.device(device or ('cuda:0' if torch.cuda.is_available() else 'cpu'))
    set_seed(seed)

    args = Args(years=years, bs=bs)
    data_path, adj_path, node_num = get_dataset_info(dataset)

    log_dir = os.path.join(log_root, model_name, dataset)
    logger = get_logger(log_dir, f'{model_name}_{dataset}', f'record_s{seed}.log')
    logger.info(f'model={model_name} dataset={dataset} seed={seed} nodes={node_num}')

    dataloader, scaler = load_dataset(data_path, args, logger)

    model, opt_factory, sched_factory, clip = MODEL_REGISTRY[model_name](
        node_num, adj_path, args, device)
    model = model.to(device)

    optimizer = opt_factory(model) if opt_factory else torch.optim.SGD(model.parameters(), lr=0.0)
    scheduler = sched_factory(optimizer) if sched_factory else None

    engine = InstrumentedEngine(
        device=device, model=model, dataloader=dataloader, scaler=scaler,
        sampler=None, loss_fn=masked_mae, lrate=1e-3, optimizer=optimizer,
        scheduler=scheduler, clip_grad_value=clip, max_epochs=max_epochs,
        patience=patience, log_dir=log_dir, logger=logger, seed=seed)

    # Trainable parameters, not param_num(): Historical Last carries a dummy
    # tensor so the optimiser has something to hold, and counting it would put
    # "1 parameter" in a table row for a model that has none. Identical to
    # param_num() for every model that actually learns.
    trainable = sum(p.nelement() for p in model.parameters() if p.requires_grad)

    record = RunRecord(model=model_name, dataset=dataset, seed=seed,
                       params=trainable, gpu_name=gpu_name(),
                       energy_method=engine._meter.method)

    results = engine.train()
    record.mae = results['mae']
    record.rmse = results['rmse']
    record.mape = results['mape']
    record.horizon_mae = results['horizon_mae']
    record.horizon_rmse = results['horizon_rmse']
    record.horizon_mape = results['horizon_mape']

    record.train_joules = engine.train_joules
    record.train_seconds = engine.train_seconds
    record.epochs_run = engine.epochs_run
    record.epoch_seconds = float(np.mean(engine.epoch_seconds)) if engine.epoch_seconds else float('nan')
    record.infer_joules_per_1k = engine.measure_inference_energy()

    # static cost, measured on a single batch
    sample = torch.randn(bs, args.seq_len, node_num, args.input_dim, device=device)
    flops, reliable = count_flops(model, sample)
    record.flops_per_sample = flops / bs if flops == flops else float('nan')
    record.flops_reliable = reliable
    record.latency_ms = measure_latency(model, sample)

    os.makedirs(results_dir, exist_ok=True)
    out = os.path.join(results_dir, f'{model_name}_{dataset}_{years}_s{seed}.json')
    record.save(out)
    logger.info(f'saved {out}')

    del model, engine, dataloader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return record

In [ ]:
%%writefile greenbench/analysis.py
"""Figures and tables for the paper.

Output is vector PDF for LaTeX plus PNG for quick viewing. These are print
figures, so they commit to the light surface only -- there is no dark variant,
deliberately.

Colour follows the validated reference palette. Two notes on why the charts
look the way they do:

  * The Pareto scatter carries identity in *text labels*, not hue. A scatter
    puts every pair of colours side by side, and no 7-hue categorical set
    survives that test; two roles (on-frontier / dominated) plus direct labels
    does survive, and reads better in print besides.
  * Aqua and yellow sit under 3:1 against the light surface, so every series
    that uses them is directly labelled. That is the documented relief for a
    contrast warning, not an oversight.
"""

import glob
import os

import matplotlib.pyplot as plt
import numpy as np

from .instrument import RunRecord, joules_to_kwh, carbon_grams, DEFAULT_GRID_INTENSITY

# --- palette (validated: see references/palette.md) ------------------------
SURFACE = '#fcfcfb'
INK_PRIMARY = '#0b0b0b'
INK_SECONDARY = '#52514e'
INK_MUTED = '#898781'
GRIDLINE = '#e1e0d9'
BASELINE = '#c3c2b7'

SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # slots 1-4
ACCENT = SERIES[0]
DOMINATED = INK_MUTED

DISPLAY_NAME = {
    'hl': 'Historical Last',
    'nlinear': 'NLinear',
    'stid': 'STID',
    'lstm': 'LSTM',
    'stgcn': 'STGCN',
    'gwnet': 'Graph WaveNet',
    'sttn': 'STTN',
}


def load_results(results_dir='results', dataset=None):
    records = [RunRecord.load(p) for p in sorted(glob.glob(os.path.join(results_dir, '*.json')))]
    if dataset:
        records = [r for r in records if r.dataset == dataset]
    return records


def _style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(BASELINE)
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=3)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_color(INK_SECONDARY)


def pareto_frontier(points):
    """Indices of non-dominated points, minimising both coordinates.

    A model is on the frontier when nothing else is both cheaper and more
    accurate. That is the whole argument of the paper in one function.
    """
    order = sorted(range(len(points)), key=lambda i: (points[i][0], points[i][1]))
    frontier, best_y = [], float('inf')
    for i in order:
        if points[i][1] < best_y:
            frontier.append(i)
            best_y = points[i][1]
    return set(frontier)


def plot_pareto(records, cost='train_joules', out='figures/pareto', annotate_savings=True):
    """Accuracy against cost, with the non-dominated set called out.

    `cost` is any RunRecord numeric field -- train_joules for the training
    story, infer_joules_per_1k or latency_ms for the deployment story. Run it
    for both; they do not always agree, and where they disagree is interesting.
    """
    records = [r for r in records if np.isfinite(getattr(r, cost)) and np.isfinite(r.mae)]
    if not records:
        raise ValueError('no records with finite cost and MAE')

    xs = [getattr(r, cost) for r in records]
    ys = [r.mae for r in records]
    frontier = pareto_frontier(list(zip(xs, ys)))

    # Historical Last does no training at all, so its cost is a true zero --
    # and a zero has no position on a log axis, so matplotlib drops the point
    # without a word. That silently deletes the cheapest model from the figure
    # whose entire argument is about cheap models. Park zeros a decade below
    # the cheapest measurable model, draw them hollow, and label them with the
    # real value so the position reads as "off-scale zero" and not as data.
    positive = [x for x in xs if x > 0]
    floor = min(positive) / 10 if positive else 1.0
    plot_xs = [x if x > 0 else floor for x in xs]

    fig, ax = plt.subplots(figsize=(6.4, 4.2), facecolor=SURFACE)
    _style_axes(ax)

    # frontier line first, so markers sit on top
    fr = sorted(frontier, key=lambda i: xs[i])
    ax.plot([plot_xs[i] for i in fr], [ys[i] for i in fr],
            color=ACCENT, linewidth=2.0, alpha=0.45, zorder=2)

    for i, r in enumerate(records):
        on = i in frontier
        is_zero = xs[i] <= 0
        color = ACCENT if on else DOMINATED
        ax.plot(plot_xs[i], ys[i], marker='o', markersize=9 if on else 8,
                color=color,
                markerfacecolor=SURFACE if is_zero else color,
                markeredgecolor=color if is_zero else SURFACE, markeredgewidth=2.0,
                linestyle='none', zorder=3)
        name = DISPLAY_NAME.get(r.model, r.model)
        ax.annotate(f'{name} (0)' if is_zero else name, (plot_xs[i], ys[i]),
                    textcoords='offset points', xytext=(9, 4),
                    fontsize=9, color=INK_PRIMARY if on else INK_SECONDARY,
                    fontweight='bold' if on else 'normal', zorder=4)

    # room for the labels, which sit to the right of their markers
    ax.margins(x=0.16, y=0.12)

    ax.set_xscale('log')
    labels = {
        'train_joules': 'Training energy (J, log scale)',
        'infer_joules_per_1k': 'Inference energy per 1000 samples (J, log scale)',
        'latency_ms': 'Inference latency (ms, log scale)',
        'flops_per_sample': 'Forward FLOPs per sample (log scale)',
        'params': 'Parameters (log scale)',
    }
    ax.set_xlabel(labels.get(cost, cost), fontsize=10, color=INK_SECONDARY)
    ax.set_ylabel('Test MAE (avg. over 12 horizons)', fontsize=10, color=INK_SECONDARY)

    # legend: two roles, always present
    handles = [
        plt.Line2D([], [], marker='o', markersize=9, color=ACCENT, linestyle='none',
                   markeredgecolor=SURFACE, markeredgewidth=2.0, label='Pareto-optimal'),
        plt.Line2D([], [], marker='o', markersize=8, color=DOMINATED, linestyle='none',
                   markeredgecolor=SURFACE, markeredgewidth=2.0, label='Dominated'),
    ]
    leg = ax.legend(handles=handles, frameon=False, fontsize=9, loc='upper right')
    for text in leg.get_texts():
        text.set_color(INK_SECONDARY)

    if annotate_savings and cost == 'train_joules':
        # Compare against the cheapest model that actually trains: a ratio
        # against Historical Last's zero is either a divide-by-zero or an
        # infinity, and neither is a sentence you can put in a figure title.
        paid = [i for i in frontier if xs[i] > 0]
        richest = max(range(len(records)), key=lambda i: xs[i])
        if paid and xs[richest] > 0:
            cheapest = min(paid, key=lambda i: xs[i])
            factor = xs[richest] / xs[cheapest]
            # Signed MAE deltas invite the wrong reading -- lower MAE is
            # better, so "+20.12 MAE" looks like a penalty when it is the
            # gain. Spell the direction out in words.
            gap = ys[cheapest] - ys[richest]
            direction = 'lower' if gap > 0 else 'higher'
            ax.set_title(
                f'{DISPLAY_NAME.get(records[richest].model, records[richest].model)} costs '
                f'{factor:,.0f}x the training energy of '
                f'{DISPLAY_NAME.get(records[cheapest].model, records[cheapest].model)} '
                f'for {abs(gap):.2f} {direction} MAE',
                fontsize=10, color=INK_PRIMARY, loc='left', pad=12)

    fig.tight_layout()
    _save(fig, out)
    return fig


def plot_horizon(records, models=None, out='figures/horizon'):
    """MAE against forecast step. Caps at four series -- see module docstring."""
    if models:
        records = [r for r in records if r.model in models]
    records = [r for r in records if r.horizon_mae]
    records = sorted(records, key=lambda r: r.mae)[:4]

    fig, ax = plt.subplots(figsize=(6.4, 4.0), facecolor=SURFACE)
    _style_axes(ax)

    for i, r in enumerate(records):
        steps = np.arange(1, len(r.horizon_mae) + 1)
        color = SERIES[i % len(SERIES)]
        ax.plot(steps, r.horizon_mae, color=color, linewidth=2.0,
                marker='o', markersize=5, markeredgecolor=SURFACE,
                markeredgewidth=1.5, label=DISPLAY_NAME.get(r.model, r.model), zorder=3)
        # direct label at the line end (also the relief for low-contrast hues)
        ax.annotate(DISPLAY_NAME.get(r.model, r.model),
                    (steps[-1], r.horizon_mae[-1]),
                    textcoords='offset points', xytext=(8, 0),
                    fontsize=9, color=INK_PRIMARY, va='center', zorder=4)

    ax.set_xlabel('Forecast horizon (15-min steps)', fontsize=10, color=INK_SECONDARY)
    ax.set_ylabel('Test MAE', fontsize=10, color=INK_SECONDARY)

    # Every line is labelled at its end, so a legend would spell the same four
    # names a second time -- and it lands in the top-left, on top of the data.
    # The gutter on the right exists for those labels, so stop the ticks at
    # the last real horizon rather than tick out into empty space.
    steps = len(records[0].horizon_mae) if records else 12
    ax.set_xlim(0.5, steps + 3)
    ax.set_xticks(range(2, steps + 1, 2))

    fig.tight_layout()
    _save(fig, out)
    return fig


def _save(fig, out):
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    fig.savefig(f'{out}.pdf', bbox_inches='tight', facecolor=SURFACE)
    fig.savefig(f'{out}.png', dpi=200, bbox_inches='tight', facecolor=SURFACE)
    print(f'wrote {out}.pdf and {out}.png')


def latex_table(records, grid_intensity=DEFAULT_GRID_INTENSITY, caption=None,
                label='tab:main', epoch_cap=100):
    """The paper's main results table.

    Sorted by training energy so the cost gradient is visible down the column,
    which is the point. FLOPs the counter could not resolve print as '--'
    rather than a fabricated number.

    Models that reach `epoch_cap` never early-stopped, so their training energy
    is a LOWER BOUND and their error is pessimistic. Those rows are daggered
    here rather than left looking like converged runs -- the Threats section
    promises the table flags them, so the table has to actually do it.
    """
    records = sorted(records, key=lambda r: (r.train_joules if np.isfinite(r.train_joules) else 0))
    caption = caption or (
        'Accuracy and cost on LargeST-SD (716 sensors, 2019, 15-min). '
        'Energy is GPU-only, measured via NVML. '
        f'Carbon assumes {grid_intensity:.0f}~gCO$_2$e/kWh. '
        f'$\\dag$~marks a model that reached the {epoch_cap}-epoch cap without '
        'early stopping: its energy is a lower bound, its error pessimistic.')

    def _sig(value, places=2):
        """Keep small numbers legible.

        The lineup spans four orders of magnitude, so a fixed 1-decimal format
        prints NLinear's 0.0035 MFLOPs as '0.0' -- which a reader takes as
        zero cost, and zero cost is precisely the claim the paper must not
        overstate. Widen the format for anything below the rounding floor.
        """
        if not np.isfinite(value):
            return '--'
        if value == 0:
            return '0'
        return f'{value:.{places}f}' if abs(value) >= 10 ** -places else f'{value:.3g}'

    rows = []
    for r in records:
        flops = '--' if not r.flops_reliable or not np.isfinite(r.flops_per_sample) \
            else _sig(r.flops_per_sample / 1e6)
        name = DISPLAY_NAME.get(r.model, r.model) + (
            '$^{\\dag}$' if np.isfinite(r.epochs_run) and r.epochs_run >= epoch_cap
            else '')
        rows.append(' & '.join([
            name,
            f'{r.params:,}',
            flops,
            f'{r.mae:.2f}',
            f'{r.rmse:.2f}',
            f'{r.mape * 100:.2f}',
            _sig(joules_to_kwh(r.train_joules) * 1000),
            _sig(carbon_grams(r.train_joules, grid_intensity)),
            _sig(r.infer_joules_per_1k),
            _sig(r.latency_ms),
        ]) + r' \\')

    return '\n'.join([
        r'\begin{table*}[t]',
        r'\centering',
        r'\caption{' + caption + '}',
        r'\label{' + label + '}',
        r'\begin{tabular}{lrrrrrrrrr}',
        r'\toprule',
        r'Model & Params & MFLOPs & MAE & RMSE & MAPE (\%) & Train (Wh) & gCO$_2$e & Infer (J/1k) & Latency (ms) \\',
        r'\midrule',
        *rows,
        r'\bottomrule',
        r'\end{tabular}',
        r'\end{table*}',
    ])

In [ ]:
import sys
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from greenbench import compat
compat.apply_all(ROOT)

## 4. Get the data**On Kaggle there is nothing to do here** -- the dataset is already mountedread-only at `/kaggle/input/largest`. Both cells below no-op and you can movestraight to section 5.**On Colab** you need an API token: kaggle.com > your profile > Settings > API.Kaggle issues two formats depending on account age -- a single bearer token(`KGAT_...`, saved to `~/.kaggle/access_token`) or the older `kaggle.json` witha username and key. The cell takes either.Put the token in Colab's **secret manager**, not in a cell: key icon in the leftsidebar > add `KAGGLE_API_TOKEN` > enable "Notebook access". A token typed intoa cell is saved inside the `.ipynb` and travels to anyone you send it to.We pull only the three files the SD subset needs -- the 2019 history, the sensormetadata and the road-network adjacency. That is still ~7.8 GB, most of it thehistory file; the full five-year archive is roughly 30 GB.

In [ ]:
import os

if IN_KAGGLE:
    print('Kaggle: dataset mounted, no token or download needed.')
    print(sorted(os.listdir(CA_DIR)))
else:
    !pip install --quiet kaggle

    from getpass import getpass
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        pass                  # secret not set, or not running on Colab
    if not token:
        token = getpass('Kaggle API token (KGAT_...): ').strip()

    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    token_path = os.path.expanduser('~/.kaggle/access_token')
    with open(token_path, 'w') as fh:
        fh.write(token.strip())
    os.chmod(token_path, 0o600)

    # Fails loudly here rather than three cells later if the token is wrong.
    !kaggle datasets files liuxu77/largest

In [ ]:
# Pull the three files we need. These names were checked against the live
# Kaggle listing on 2026-08-02; if the cell above ever prints something
# different, correct it here rather than downloading the whole archive.
# ca_his_raw_2019.h5 is 7.2 GB, so this cell is the slow one.
import glob, os, zipfile

YEAR = '2019'
NEEDED = [f'ca_his_raw_{YEAR}.h5', 'ca_meta.csv', 'ca_rn_adj.npy']

if not IN_KAGGLE:
    os.makedirs(CA_DIR, exist_ok=True)
    os.chdir(CA_DIR)

    for fname in NEEDED:
        if os.path.exists(fname):
            print(f'have {fname}')
            continue
        !kaggle datasets download liuxu77/largest -f {fname} --force
        # Single-file downloads sometimes arrive zipped. Both globs can match
        # the same archive, so dedupe them -- otherwise the second pass tries
        # to open the zip the first pass already extracted and deleted, and
        # the whole loop dies *after* a 7 GB download has succeeded.
        for z in sorted(set(glob.glob(f'{fname}.zip')) | set(glob.glob('*.zip'))):
            if not os.path.exists(z):
                continue
            with zipfile.ZipFile(z) as zf:
                zf.extractall('.')
            os.remove(z)

    os.chdir(ROOT)

missing = [f for f in NEEDED if not os.path.exists(os.path.join(CA_DIR, f))]
assert not missing, f'missing from {CA_DIR}: {missing}'
!ls -lh {CA_DIR}

## 5. Build the SD subsetUpstream does this by loading the whole California frame and slicing District11 out of it -- about 7 GB resident, which a free Colab instance cannot hold.`greenbench.prepare` reads the file in row blocks and keeps only the 716columns it needs.

In [ ]:
from greenbench.prepare import build_sd_subset

# ca_dir is read-only on Kaggle -- build_sd_subset only reads from it, and
# every write goes to sd_dir under the working tree.
sd = build_sd_subset(ca_dir=CA_DIR, sd_dir=f'{ROOT}/data/sd', year=YEAR,
                     resample='15min')
print(sd.index[0], '->', sd.index[-1])
sd.iloc[:5, :6]

In [ ]:
# Window the series into (12 in -> 12 out) samples with 60/20/20 splits.
# Writes data/sd/2019/{his.npz, idx_train.npy, idx_val.npy, idx_test.npy}.
import os
os.chdir(f'{ROOT}/data')
!python generate_data_for_training.py --dataset sd --years {YEAR}
os.chdir(ROOT)

import numpy as np
ptr = np.load(f'data/sd/{YEAR}/his.npz')
print('data', ptr['data'].shape, '| mean', float(ptr['mean']), '| std', float(ptr['std']))
for split in ['train', 'val', 'test']:
    print(split, np.load(f'data/sd/{YEAR}/idx_{split}.npy').shape)

## 6. Run the sweepOne code path builds every model, so batch size, input dimension, loss,splits and early-stopping rule are identical across the lineup -- the onlything that varies is the architecture. Per-model hyperparameters are theupstream defaults; no baseline has been tuned down.Results are written to `results/*.json` after each run, so a disconnect costsyou one model rather than the sweep.

In [ ]:
import os
from greenbench.runner import run_experiment, MODEL_REGISTRY

QUICK_SWEEP = True   # True: cheap models only, ~30 min. False: everything.
SEEDS = [2023]       # add 2024, 2025 before the camera-ready
MAX_EPOCHS = 100

# Upstream LargeST uses 30. Dropping it to 15 cuts off slow-converging models
# -- which is exactly the expensive class this paper argues against -- so the
# protocol would be quietly biased toward its own conclusion. Keep 30 and pay
# the runtime; every model in the table must use the same value, including the
# ones run one at a time in section 6b.
PATIENCE = 30

CHEAP = ['hl', 'nlinear', 'stid', 'lstm']
FULL = CHEAP + ['stgcn', 'gwnet', 'sttn']
models = []

print('will run:', models)
print('results ->', RESULTS)

In [ ]:
import traceback

records = []
for seed in SEEDS:
    for name in models:
        # Already-finished models are skipped, so re-running this cell after a
        # disconnect resumes the sweep instead of restarting it.
        done = os.path.join(RESULTS, f'{name}_SD_{YEAR}_s{seed}.json')
        if os.path.exists(done):
            print(f'skip {name} seed={seed} -- already in {RESULTS}')
            continue

        print(f'\n{"=" * 60}\n{name}  seed={seed}\n{"=" * 60}')
        try:
            rec = run_experiment(name, dataset='SD', years=YEAR, seed=seed,
                                 bs=64, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                                 results_dir=RESULTS, log_root=LOGS)
            records.append(rec)
            print(f'  MAE {rec.mae:.3f} | train {rec.train_joules / 1000:.1f} kJ '
                  f'({rec.epochs_run} epochs) | {rec.latency_ms:.1f} ms | {rec.params:,} params')
        except Exception:
            # One failed model should not end the sweep.
            traceback.print_exc()

### 6b. Run selected modelsThe heavy models do not fit in one session, so run them a few at a time withthis cell instead of the one above. Edit `TODO`, run, download the JSONs.Anything already present in `RESULTS` is skipped, so cancelling and re-runningresumes rather than restarts -- **within a live session**. A session that endstakes `/kaggle/working` with it, and then nothing is left to skip; that is whatthe downloaded JSONs are for.

In [ ]:
import os, traceback

# This cell stands alone, so it is the one people jump straight to after a
# restart -- at which point the modules are not on disk yet and the import
# below fails with a bare ModuleNotFoundError that says nothing useful.
missing = [n for n in ('ROOT', 'RESULTS', 'LOGS', 'YEAR') if n not in globals()]
assert not missing, (
    f'{missing} undefined -- sections 1-5 have not run in this session. '
    'Select the section 6 cell and use Run > Run Before, then come back here.')

from greenbench.runner import run_experiment, MODEL_REGISTRY

TODO = ['hl', 'nlinear', 'stid']
SEED = 2023
MAX_EPOCHS = 100

PATIENCE = 30         # same value as section 6 -- see the note there

unknown = [m for m in TODO if m not in MODEL_REGISTRY]
assert not unknown, f'unknown: {unknown}; have {sorted(MODEL_REGISTRY)}'

for name in TODO:
    done = os.path.join(RESULTS, f'{name}_SD_{YEAR}_s{SEED}.json')
    if os.path.exists(done):
        print(f'skip {name} -- already in {RESULTS}')
        continue

    print(f'\n{"=" * 60}\n{name}  seed={SEED}  patience={PATIENCE}\n{"=" * 60}')
    try:
        rec = run_experiment(name, dataset='SD', years=YEAR, seed=SEED,
                             bs=64, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                             results_dir=RESULTS, log_root=LOGS)
        print(f'  MAE {rec.mae:.3f} | train {rec.train_joules / 1000:.1f} kJ '
              f'({rec.epochs_run} ep) | infer/1k {rec.infer_joules_per_1k:.3f} J '
              f'| {rec.latency_ms:.2f} ms | {rec.params:,} params')
    except Exception:
        traceback.print_exc()

print('\nfiles in', RESULTS)
print(sorted(os.listdir(RESULTS)))

## 7. Figures and table

In [ ]:
import traceback

from greenbench import analysis

recs = analysis.load_results(RESULTS, dataset='SD')
print(f'{len(recs)} runs loaded:', sorted(r.model for r in recs))


def _try(label, fn, *a, **kw):
    """Run a figure/table step without letting it fail the notebook.

    In a committed (Save & Run All) Kaggle run an uncaught exception aborts the
    version, and hours of finished training go with it. Every step below is
    derived output that can be regenerated locally from the JSONs in seconds --
    so nothing here is worth losing a run over. plot_pareto in particular
    raises outright when no record has a finite cost, which is the normal case
    for a single-model commit whose inference meter returned NaN.
    """
    try:
        return fn(*a, **kw)
    except Exception:
        print(f'-- {label} failed (results are still saved); traceback follows')
        traceback.print_exc()


_try('pareto/train', analysis.plot_pareto, recs, cost='train_joules',
     out=f'{FIGURES}/pareto_train')
_try('pareto/infer', analysis.plot_pareto, recs, cost='infer_joules_per_1k',
     out=f'{FIGURES}/pareto_infer')
_try('horizon', analysis.plot_horizon, recs, out=f'{FIGURES}/horizon')

In [ ]:
table = _try('latex_table', analysis.latex_table, recs)
if table:
    print(table)
    with open(f'{FIGURES}/main_table.tex', 'w') as fh:
        fh.write(table)

In [ ]:
# On Kaggle everything under /kaggle/working is already saved with the notebook
# version -- use "Save Version > Save & Run All" and collect it from the Output
# tab. On Colab the results are in Drive; this is a convenience copy.
import os, shutil

# Stage in /tmp: zipping a directory into itself makes the archive walk its own
# growing output.
staging = '/tmp/icaisd_bundle'
shutil.rmtree(staging, ignore_errors=True)
os.makedirs(staging)
for src in (RESULTS, FIGURES):
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(staging, os.path.basename(src)))

archive = shutil.copy(shutil.make_archive('/tmp/icaisd_results', 'zip', staging), OUT)
print('wrote', archive)

if not IN_KAGGLE:
    from google.colab import files
    files.download(archive)

## Before this becomes a paperThings that are deliberately unfinished here:- **Seeds.** One seed is a pilot, not a result. Run at least three and report  mean +/- std; a 0.05 MAE gap across models means nothing next to seed noise.- **Carbon intensity.** `instrument.DEFAULT_GRID_INTENSITY` is a placeholder.  Replace it with a sourced CAISO figure and cite it -- LargeST is California  data, so a California grid factor is the defensible choice.- **Energy scope.** We measure GPU draw only: no CPU, no DRAM, no PUE. Say so.  It makes the numbers a lower bound rather than an overstatement.- **Shared hardware.** Colab hosts are multi-tenant, so energy readings carry  noise you do not control. Report the GPU model, run the sweep in one session  where possible, and treat order-of-magnitude gaps as the finding rather than  10% differences.- **Excluded baselines.** STGODE (hours of DTW preprocessing) and the six  models needing custom engines (DCRNN, AGCRN, ASTGCN, DGCRN, DSTAGNN,  D2STGNN) are not in the lineup. State that plainly; do not imply the sweep  is exhaustive.